# Login form with validation

This notebook builds a login form with `TextField` inputs and `@checks` validation rules. The sign-in button is disabled until the email and password pass their checks. Submitting sends a `login` action whose context carries the email.

Run the surface cell, then interact with the form. The callback updates the status after the click. Run the last cell to inspect the action.

In [ ]:
from renderer_setup import new_surface

surface = new_surface("login-form")
surface.send(
    [
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": "login-form",
                "value": {"email": "", "password": "", "status": "Enter your details to sign in"},
            },
        },
        {
            "version": "1.0",
            "updateComponents": {
                "surfaceId": "login-form",
                "components": [
                    {"id": "root", "component": "Card", "child": "main_column"},
                    {
                        "id": "main_column",
                        "component": "Column",
                        "children": ["title", "email_field", "password_field", "login_btn", "status"],
                    },
                    {"id": "title", "component": "Text", "text": "## Welcome back"},
                    {
                        "id": "email_field",
                        "component": "TextField",
                        "label": "Email",
                        "value": {"@path": "/email"},
                        "@checks": [
                            {
                                "condition": {"@call": "required", "args": {"value": {"@path": "/email"}}},
                                "message": "Email is required",
                            },
                            {
                                "condition": {"@call": "email", "args": {"value": {"@path": "/email"}}},
                                "message": "Please enter a valid email address",
                            },
                        ],
                    },
                    {
                        "id": "password_field",
                        "component": "TextField",
                        "label": "Password",
                        "variant": "obscured",
                        "value": {"@path": "/password"},
                        "@checks": [
                            {
                                "condition": {"@call": "required", "args": {"value": {"@path": "/password"}}},
                                "message": "Password is required",
                            },
                            {
                                "condition": {
                                    "@call": "length",
                                    "args": {"value": {"@path": "/password"}, "min": 8},
                                },
                                "message": "Password must be at least 8 characters long",
                            },
                        ],
                    },
                    {
                        "id": "login_btn",
                        "component": "Button",
                        "child": "login_btn_text",
                        "variant": "primary",
                        "@checks": [
                            {
                                "condition": {
                                    "@call": "and",
                                    "args": {
                                        "values": [
                                            {"@call": "email", "args": {"value": {"@path": "/email"}}},
                                            {
                                                "@call": "length",
                                                "args": {"value": {"@path": "/password"}, "min": 8},
                                            },
                                        ]
                                    },
                                },
                                "message": "Please fix errors before signing in",
                            }
                        ],
                        "action": {
                            "event": {
                                "name": "login",
                                "context": {"email": {"@path": "/email"}},
                            }
                        },
                    },
                    {"id": "login_btn_text", "component": "Text", "text": "Sign in"},
                    {"id": "status", "component": "Text", "text": {"@path": "/status"}, "variant": "caption"},
                ],
            },
        },
    ]
)

action_data = None


def handle_action(action):
    global action_data
    action_data = action.model_dump(by_alias=True, exclude_unset=True)
    surface.send(
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": "login-form",
                "path": "/status",
                "value": f"Signed in as {action_data['action']['context']['email']}",
            },
        }
    )


surface.on_action(handle_action)
surface

After the `login` action, inspect its payload:

In [ ]:
action_data